# Semiconductor Patent RAG — Databricks

Databricks-ready migration of the local Jupyter RAG. Configure `VOLUME_ROOT` before running.


In [ ]:
from pathlib import Path
import os, re, requests
import numpy as np
import pandas as pd

VOLUME_ROOT = Path("/Volumes/<catalog>/<schema>/<volume>/semiconductor_patents")
NORMALIZED_FILE = VOLUME_ROOT / "normalized_patents.csv"
CHUNK_TABLE = "semiconductor_patent_chunks"
EMBED_TABLE = "semiconductor_patent_embeddings"
CHUNK_SIZE, CHUNK_OVERLAP = 1500, 250
EMBED_MODEL, LLM_MODEL = "nomic-embed-text", "qwen3:8b"
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_EMBED_URL = f"{OLLAMA_BASE_URL}/api/embeddings"
OLLAMA_GENERATE_URL = f"{OLLAMA_BASE_URL}/api/generate"

if not VOLUME_ROOT.exists():
    raise FileNotFoundError(f"Update VOLUME_ROOT: {VOLUME_ROOT}")
patent_files = sorted(VOLUME_ROOT.rglob("*.txt"))
print("Patent files:", len(patent_files))


In [ ]:
def parse_patent_file(path):
    text = path.read_text(encoding="utf-8", errors="ignore")
    def get(pattern, default=""):
        m = re.search(pattern, text, re.IGNORECASE)
        return m.group(1).strip() if m else default
    a = re.search(r"ABSTRACT\s*\n=+\s*\n(.*?)(?=\n(?:DESCRIPTION|DETAILED DESCRIPTION)\s*\n=+)", text, re.DOTALL | re.IGNORECASE)
    d = re.search(r"(?:DESCRIPTION|DETAILED DESCRIPTION)\s*\n=+\s*\n(.*)", text, re.DOTALL | re.IGNORECASE)
    return {
        "document_id": get(r"DOCUMENT[_ ]ID:\s*(\S+)", path.stem.replace("patent_", "")),
        "category": get(r"CATEGORY:\s*(\S+)"),
        "cpc_section": get(r"CPC(?:[_ ]SECTION)?:\s*(\S+)"),
        "source": get(r"(?:SOURCE|DATASET):\s*(\S+)", "BIGPATENT"),
        "abstract": a.group(1).strip() if a else "",
        "description": d.group(1).strip() if d else "",
    }

normalized = pd.DataFrame(parse_patent_file(p) for p in patent_files)
for c in ["document_id","category","cpc_section","source","abstract","description"]:
    normalized[c] = normalized[c].fillna("").astype(str).str.strip()
normalized["abstract_words"] = normalized["abstract"].str.split().str.len()
normalized["description_words"] = normalized["description"].str.split().str.len()
normalized.to_csv(NORMALIZED_FILE, index=False)
print("Records:", len(normalized), "Unique patents:", normalized.document_id.nunique())
display(normalized.head())


In [ ]:
def chunk_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    if not text: return []
    chunks, start = [], 0
    while start < len(text):
        end = start + size
        piece = text[start:end].strip()
        if piece: chunks.append(piece)
        if end >= len(text): break
        start = end - overlap
    return chunks

rows = []
for _, p in normalized.iterrows():
    cid = 0
    for section, text in [("abstract", p.abstract), ("description", p.description)]:
        for piece in chunk_text(text):
            rows.append({"chunk_id":cid,"document_id":p.document_id,"category":p.category,"cpc_section":p.cpc_section,"source":p.source,"section":section,"text":piece})
            cid += 1
chunks_df = pd.DataFrame(rows)
spark.createDataFrame(chunks_df).write.format("delta").mode("overwrite").saveAsTable(CHUNK_TABLE)
print("Chunks:", len(chunks_df), "Patents:", chunks_df.document_id.nunique())


In [ ]:
def embed_text(text):
    r = requests.post(OLLAMA_EMBED_URL, json={"model":EMBED_MODEL,"prompt":text}, timeout=120)
    r.raise_for_status()
    return r.json()["embedding"]

embeddings = np.asarray([embed_text(x) for x in chunks_df.text], dtype=np.float32)
embeddings /= np.maximum(np.linalg.norm(embeddings, axis=1, keepdims=True), 1e-12)
embedding_rows = [{"chunk_id":int(i),"document_id":chunks_df.iloc[i].document_id,"embedding":embeddings[i].tolist()} for i in range(len(chunks_df))]
spark.createDataFrame(pd.DataFrame(embedding_rows)).write.format("delta").mode("overwrite").saveAsTable(EMBED_TABLE)
print("Embedding matrix:", embeddings.shape)


In [ ]:
def retrieve_patents(query, top_k=5, candidate_k=30):
    q = np.asarray(embed_text(query), dtype=np.float32)
    q /= max(np.linalg.norm(q), 1e-12)
    scores = embeddings @ q
    selected, seen = [], set()
    for idx in np.argsort(scores)[::-1][:candidate_k]:
        pid = chunks_df.iloc[idx].document_id
        if pid in seen: continue
        seen.add(pid); selected.append(idx)
        if len(selected) == top_k: break
    result = chunks_df.iloc[selected].copy().reset_index(drop=True)
    result["similarity"] = scores[selected]
    return result

def ask_ollama(prompt):
    r = requests.post(OLLAMA_GENERATE_URL, json={"model":LLM_MODEL,"prompt":prompt,"stream":False}, timeout=120)
    r.raise_for_status()
    return r.json()["response"]

question = "What techniques are used to reduce impedance in semiconductor packaging?"
results = retrieve_patents(question)
context = "\n\n".join(f"Patent ID: {r.document_id}\nCategory: {r.category}\nText: {r.text}" for _, r in results.iterrows())
prompt = f"""You are a semiconductor patent research assistant. Use ONLY the evidence below. Cite patent IDs as [Patent: ID]. Do not invent facts. This corpus contains no claims, so do not make legal or claim-coverage conclusions.\n\nEVIDENCE:\n{context}\n\nQUESTION:\n{question}"""
print(ask_ollama(prompt))


## Migration notes

- Mac `/Users/jamesjr/...` paths are removed.
- Old BGE/SentenceTransformer and Jupyter widget outputs are removed.
- Chunks and embeddings are persisted as Delta tables.
- **Important:** Databricks `localhost:11434` is not the Ollama process on your Mac. `OLLAMA_BASE_URL` must point to a reachable endpoint, or we should replace Ollama with Databricks Model Serving.
- The corpus has no claims, so the RAG does not make claim-coverage or legal conclusions.
